# Ejercicio 1: resumen por servicio, a mano y con pandas

Prepara el notebook con lo de siempre (sys.path, autoreload, from src import loader) y carga el dataset completo con tu función loader.cargar_todos(...) en una variable df_todo.

In [40]:
import pandas as pd ## Para tablas y dataframes
from pathlib import Path ## Para manejar rutas y ficheros
import sys ## Esta TAMBIÉN hay que importarla!

sys.path.append("../..") ## Para que los "import" los busque también en la carpeta del proyecto, donde se encuentra loader.py

## Cargamos la extensión autoreload al notebook (comando de Jupyter)
%load_ext autoreload 
## Activa la extensión. Ahora, antes de ejecutar cada celda, comprueba si hay algún módulo que ha cambiado y lo recarga. Si no, cargaría "loader" pero se quedaría con esa 1 versión cargada en memoria rápida.
%autoreload 2 
## NO PODEMOS PONERLE COMENTARIOS A ESTAS 2 SENTENCIAS! (las de arriba)

from src import loader ## Ahora sí podemos importar nuestro propio modulo.

df = loader.cargar_todos("../../data/raw/5G-production-dataset")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
df.shape ## Vale, todo correcto.
df.head()

,Timestamp,Longitude,Latitude,Speed,Operatorname,CellID,NetworkMode,RSRP,RSRQ,SNR,...,PINGLOSS,CELLHEX,NODEHEX,LACHEX,RAWCELLID,NRxRSRP,NRxRSRQ,servicio,movilidad,fichero
0,2019-11-28 07:27:57,-8.388193,51.935608,0,B,12,5G,-102.0,-10.0,8.0,...,NaN,C,A81B,9CBA,11016972,-102.0,-1.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
1,2019-11-28 07:27:57,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,8.0,...,NaN,C,A81B,9CBA,11016972,-102.0,-1.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
2,2019-11-28 07:27:58,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,8.0,...,NaN,C,A81B,9CBA,11016972,-102.0,-1.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
3,2019-11-28 07:27:59,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,3.0,...,NaN,C,A81B,9CBA,11016972,-101.0,-3.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv
4,2019-11-28 07:28:00,-8.388269,51.935542,1,B,12,5G,-102.0,-10.0,3.0,...,NaN,C,A81B,9CBA,11016972,-101.0,-3.0,Amazon_Prime,Driving,B_2019.11.28_07.27.57.csv


## - A) Python + pandas. Recorre con un bucle for los servicios distintos que hay en df_todo. Para cada uno, filtra sus filas y guarda en un diccionario resumen el número de muestras y la mediana de RSRP.

**Mi versión:**

In [49]:
df.columns ## Primero veo el nombre de la columna servicios: NetworkMode.
df["NetworkMode"].unique() ## Vemos qué servicios hay: ['5G', 'LTE', 'HSUPA', 'UMTS', 'HSPA+', 'HSDPA', 'EDGE', 'GPRS']
servicios = df.groupby("servicio")["RSRP"] ## Devuelve una lista de tuplas de "(nombre del servicio, su dataframe completo)".
                                           ## En este caso, su dataframe solo tiene la columna RSRP, porque es la que interesa.
resumen_RSRP = {} ## Esto es un DICCIONARIO VACÍO.
for servicio, df_servicio in servicios:
    muestras_RSRP = df_servicio.count() ## NO podemos poner .size porque cuenta los valores nulos
    mediana_RSRP = df_servicio.median()
    resumen_RSRP[servicio] = [muestras_RSRP, mediana_RSRP]

print(resumen_RSRP) ## Así se ve, pero feo.

for clave, valor in resumen_RSRP.items(): ## La función ".items()" nos da las parejas clave-valor.
    print("Servicio " + clave + ":")
    print("\tMuestras RSRP: " + str(valor[0]) + ".")
    print("\tValor RSRP mediano: " + str(valor[1]) + " dB.")  



{'Amazon_Prime': [np.int64(72602), np.float64(-97.0)], 'Download': [np.int64(43205), np.float64(-93.0)], 'Netflix': [np.int64(72831), np.float64(-91.0)]}
Servicio Amazon_Prime:
	Muestras RSRP: 72602.
	Valor RSRP mediano: -97.0 dB.
Servicio Download:
	Muestras RSRP: 43205.
	Valor RSRP mediano: -93.0 dB.
Servicio Netflix:
	Muestras RSRP: 72831.
	Valor RSRP mediano: -91.0 dB.


**Versión Claude:**

In [ ]:
resumen = {}
for servicio in df["servicio"].unique():
    rsrp = df.loc[df["servicio"] == servicio, "RSRP"]
    resumen[servicio] = {"muestras": rsrp.count(), "mediana": rsrp.median()} ## Aquí lo que estamos haciendo es un nuevo diccionario dentro del 
                                                                             ## diccionario!

resumen



{'Amazon_Prime': {'muestras': np.int64(72602), 'mediana': np.float64(-97.0)},
 'Download': {'muestras': np.int64(43205), 'mediana': np.float64(-93.0)},
 'Netflix': {'muestras': np.int64(72831), 'mediana': np.float64(-91.0)}}

## B) Python. Recorre el diccionario e imprime una línea por servicio con una f-string.

In [ ]:
resumen_RSRP

for clave, valor in resumen_RSRP.items():
    print(f"{clave}:\n\tMuestras RSRP = {valor[0]}\n\tMediana RSRP = {valor[1]}") ## bieeeeen bien. SIEMPRE RECORDAR CON LOS DICCIONARIOS
                                                                                  ## QUE SE USARÁ EL ".items()" para recorrerlos en un for.

Amazon_Prime:
	Muestras RSRP = 72602
	Mediana RSRP = -97.0
Download:
	Muestras RSRP = 43205
	Mediana RSRP = -93.0
Netflix:
	Muestras RSRP = 72831
	Mediana RSRP = -91.0


## C) Pandas. Consigue los mismos números en una sola línea con groupby, y comprueba que coinciden con los de tu diccionario.

In [ ]:
## En este caso NO guardamos en un diccionario, sino directamente trabajar en pandas y dataframe (lo que estaba haciendo hasta ahora).
## Lo que hace es agrupar por servicio, coger la columna RSRP y aplicarle VARIAS FUNCIONES A LA VEZ que se distribuirán en distintas columnas.
resultados_dataframe = df.groupby("servicio")["RSRP"].agg(["count", "median"])
resultados_dataframe

,count,median
servicio,,
Amazon_Prime,72602,-97.0
Download,43205,-93.0
Netflix,72831,-91.0


### d) Ficheros por servicio
- Con `Path` y `rglob`, recorre todos los CSV del dataset y cuenta **cuántos ficheros hay de cada servicio**, guardándolo en un diccionario (ej. `{"Download": 16, ...}`).
- Imprime el resultado con una f-string por línea.
- Carga **un** fichero de Netflix con `loader.cargar_csv()` y comprueba su `shape` y el tipo de `Timestamp`.


In [ ]:
''' Parte 1 '''

from pathlib import Path
ruta = Path("../../data/raw/5G-production-dataset")

rutas_dic = {}

for file in ruta.rglob("*.csv"):

    ## Saco el valor del servicio de cada csv
    servicio = file.relative_to(ruta).parts[0] 

    ## Creo una lista vacía para cada servicio la primera vez que este aparezca
    if servicio not in rutas_dic:
        rutas_dic[servicio] = []

    ## Según el servicio, añado un elemento a su lista.
    match servicio:
        case "Amazon_Prime":
            rutas_dic[servicio].append(file)
        case "Netflix":
            rutas_dic[servicio].append(file)
        case "Download":
            rutas_dic[servicio].append(file)

ficheros_por_servicio = {"Amazon_Prime": len(rutas_dic["Amazon_Prime"]), "Netflix": len(rutas_dic["Netflix"]), "Download": len(rutas_dic["Download"])}

''' Parte 2 '''
print(
    f"Ficheros por servicio:\n\tAmazon_Prime = {ficheros_por_servicio["Amazon_Prime"]}"
    f"\n\tNetflix = {ficheros_por_servicio["Netflix"]}"
    f"\n\tDownload = {ficheros_por_servicio["Download"]}"         
)

''' Parte 3 '''
import sys
sys.path.append("../..")
from src import loader

ruta = Path("../../data/raw/5G-production-dataset")
for file in ruta.rglob("*.csv"):
    if "Netflix" in file.parts:
        netflix_csv = file
        break

netflix_csv = loader.cargar_csv(netflix_csv)

netflix_csv.shape
netflix_csv.dtypes ## Listo

Ficheros por servicio:
	Amazon_Prime = 29
	Netflix = 33
	Download = 21


Timestamp       datetime64[us]
Longitude              float64
Latitude               float64
Speed                    int64
Operatorname               str
CellID                   int64
NetworkMode                str
RSRP                   float64
RSRQ                     int64
SNR                    float64
CQI                    float64
RSSI                   float64
DL_bitrate               int64
UL_bitrate               int64
State                      str
PINGAVG                float64
PINGMIN                float64
PINGMAX                float64
PINGSTDEV              float64
PINGLOSS               float64
CELLHEX                    str
NODEHEX                    str
LACHEX                     str
RAWCELLID                int64
NRxRSRP                float64
NRxRSRQ                float64
dtype: object

### e) Mala cobertura: conduciendo vs. estático
- Con el dataset completo (`loader.cargar_todos()`), calcula el **% de muestras con RSRP < −100 dBm** para cada valor de `movilidad`.
- Hazlo con un **bucle `for`** sobre los valores únicos y una **máscara**, sin `groupby`.
- Imprime una línea por movilidad, con 1 decimal.
- ¿Qué esperabas antes de ejecutarlo? ¿Cuadra?

In [ ]:
## HACER EJERCICIO

### f) Throughput por servicio y movilidad
- Crea la columna `DL_Mbps` (suponiendo `DL_bitrate` en kbps).
- Calcula la **mediana de `DL_Mbps`** para cada combinación de `servicio` y `movilidad`, en una línea, y ordénala de mayor a menor.
- ¿Qué combinación tiene más throughput? ¿Tiene sentido según el tipo de tráfico (descarga de ficheros vs. vídeo)?

In [ ]:
## HACER EJERCICIO